In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Medium_accounts.csv
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/LI-Small_Trans.csv
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Large_accounts.csv
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Large_Trans.csv
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Medium_Trans.csv
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/LI-Medium_accounts.csv
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Patterns.txt
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Medium_Patterns.txt
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_accounts.csv
/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-la

In [2]:
import pandas as pd

BASE = "/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml"

df = pd.read_csv(f"{BASE}/HI-Small_Trans.csv", dtype={"Account": str, "Account.1": str})
print(df.shape)
print(df["Is Laundering"].value_counts())
print("Time range:", df["Timestamp"].min(), "→", df["Timestamp"].max())
print("Unique banks:", pd.concat([df["From Bank"], df["To Bank"]]).nunique())
df.head()

(5078345, 11)
Is Laundering
0    5073168
1       5177
Name: count, dtype: int64
Time range: 2022/09/01 00:00 → 2022/09/18 16:18
Unique banks: 30470


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [3]:
with open(f"{BASE}/HI-Small_Patterns.txt") as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i > 30:
            break

BEGIN LAUNDERING ATTEMPT - FAN-OUT:  Max 16-degree Fan-Out
2022/09/01 00:06,021174,800737690,012,80011F990,2848.96,Euro,2848.96,Euro,ACH,1
2022/09/01 04:33,021174,800737690,020,80020C5B0,8630.40,Euro,8630.40,Euro,ACH,1
2022/09/01 09:14,021174,800737690,020,80006A5E0,35642.49,Yuan,35642.49,Yuan,ACH,1
2022/09/01 09:56,021174,800737690,00220,8007A5B70,5738987.96,US Dollar,5738987.96,US Dollar,ACH,1
2022/09/01 11:28,021174,800737690,001244,80093C0D0,7254.53,US Dollar,7254.53,US Dollar,ACH,1
2022/09/01 13:13,021174,800737690,00513,80078E200,6990.87,US Dollar,6990.87,US Dollar,ACH,1
2022/09/01 14:11,021174,800737690,020,80066B990,12536.92,Euro,12536.92,Euro,ACH,1
2022/09/02 15:40,021174,800737690,00410,8002CC310,3511.82,Euro,3511.82,Euro,ACH,1
2022/09/02 21:23,021174,800737690,01292,8004030A0,16135.09,US Dollar,16135.09,US Dollar,ACH,1
2022/09/02 23:10,021174,800737690,01601,800578800,12183.28,US Dollar,12183.28,US Dollar,ACH,1
2022/09/03 09:29,021174,800737690,001,800AAF0B0,15197.45,US Doll

In [4]:
import re

attempts, current = [], None
with open(f"{BASE}/HI-Small_Patterns.txt") as f:
    for line in f:
        line = line.strip()
        if line.startswith("BEGIN LAUNDERING ATTEMPT"):
            ptype = line.split("-", 1)[1].strip() if "-" in line else "UNKNOWN"
            current = {"type": ptype, "rows": []}
        elif line.startswith("END LAUNDERING ATTEMPT"):
            attempts.append(current); current = None
        elif current is not None and line:
            current["rows"].append(line.split(","))

print("Attempts found:", len(attempts))
print(pd.Series([a["type"] for a in attempts]).value_counts())

# Flatten pattern transactions
rows = []
for cid, a in enumerate(attempts):
    for r in a["rows"]:
        rows.append([cid, a["type"]] + r[:5] + [r[7]])  # timestamp, from bank, acct, to bank, acct, amount paid
pat = pd.DataFrame(rows, columns=["campaign_id", "type", "Timestamp", "From Bank", "Account",
                                  "To Bank", "Account.1", "Amount Paid"])
pat["From Bank"] = pat["From Bank"].astype(int)
pat["To Bank"] = pat["To Bank"].astype(int)
pat["Amount Paid"] = pat["Amount Paid"].astype(float)

# Match against the transaction file
keys = ["Timestamp", "From Bank", "Account", "To Bank", "Account.1", "Amount Paid"]
merged = pat.merge(df[keys + ["Is Laundering"]], on=keys, how="left", indicator=True)

print("\nPattern transactions:", len(pat))
print("Matched in Trans.csv:", (merged["_merge"] == "both").sum())
print("Unmatched:", (merged["_merge"] == "left_only").sum())
print("Total laundering rows in Trans.csv:", df["Is Laundering"].sum())

Attempts found: 370
BIPARTITE                               49
SCATTER-GATHER                          44
STACK                                   43
CYCLE:  Max 2 hops                      14
RANDOM:  Max 1 hops                     13
                                        ..
GATHER-SCATTER:  Max 6-degree Fan-In     1
FAN-OUT:  Max 14-degree Fan-Out          1
GATHER-SCATTER:  Max 9-degree Fan-In     1
FAN-OUT:  Max 15-degree Fan-Out          1
GATHER-SCATTER:  Max 5-degree Fan-In     1
Name: count, Length: 65, dtype: int64

Pattern transactions: 3209
Matched in Trans.csv: 3209
Unmatched: 0
Total laundering rows in Trans.csv: 5177


In [5]:
import numpy as np

pat["base_type"] = pat["type"].str.split(":").str[0].str.strip()
pat["ts"] = pd.to_datetime(pat["Timestamp"])
pat["src"] = pat["From Bank"].astype(str) + "_" + pat["Account"]
pat["dst"] = pat["To Bank"].astype(str) + "_" + pat["Account.1"]

camp = pat.groupby("campaign_id").agg(
    base_type=("base_type", "first"),
    n_txn=("ts", "size"),
    start=("ts", "min"),
    end=("ts", "max"),
    n_banks=("From Bank", lambda s: len(set(s) | set(pat.loc[s.index, "To Bank"]))),
)
camp["n_accounts"] = pat.groupby("campaign_id").apply(lambda g: len(set(g.src) | set(g.dst)))
camp["duration_h"] = (camp["end"] - camp["start"]).dt.total_seconds() / 3600

print(camp["base_type"].value_counts(), "\n")
print(camp[["n_txn", "n_accounts", "n_banks", "duration_h"]].describe().round(1), "\n")
print("Campaigns starting per day:")
print(camp["start"].dt.date.value_counts().sort_index())

# Overall activity per day (spots a sparse tail at the end)
df["ts"] = pd.to_datetime(df["Timestamp"])
print("\nTransactions per day:")
print(df["ts"].dt.date.value_counts().sort_index())

base_type
CYCLE             54
GATHER-SCATTER    51
BIPARTITE         49
FAN-OUT           48
SCATTER-GATHER    44
STACK             43
RANDOM            41
FAN-IN            40
Name: count, dtype: int64 

       n_txn  n_accounts  n_banks  duration_h
count  370.0       370.0    370.0       370.0
mean     8.7         9.7      9.0        67.5
std      7.7         8.4      7.9        46.3
min      1.0         2.0      1.0         0.0
25%      2.0         3.0      2.0        32.8
50%      6.5         8.0      7.0        74.7
75%     13.0        14.0     13.0        90.9
max     32.0        45.0     44.0       202.3 

Campaigns starting per day:
start
2022-09-01    38
2022-09-02    38
2022-09-03    32
2022-09-04    33
2022-09-05    38
2022-09-06    41
2022-09-07    37
2022-09-08    27
2022-09-09    42
2022-09-10    37
2022-09-11     2
2022-09-12     5
Name: count, dtype: int64


/tmp/ipykernel_16/2207266864.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  camp["n_accounts"] = pat.groupby("campaign_id").apply(lambda g: len(set(g.src) | set(g.dst)))



Transactions per day:
ts
2022-09-01    1114921
2022-09-02     754449
2022-09-03     207382
2022-09-04     207430
2022-09-05     482650
2022-09-06     482089
2022-09-07     482751
2022-09-08     482773
2022-09-09     654467
2022-09-10     208325
2022-09-11        396
2022-09-12        281
2022-09-13        184
2022-09-14        121
2022-09-15         46
2022-09-16         46
2022-09-17         23
2022-09-18         11
Name: count, dtype: int64


In [6]:
cut = pd.Timestamp("2022-09-11")

# 1. How laundering-heavy is the tail?
tail = df[df["ts"] >= cut]
print("Tail rows:", len(tail), "| laundering:", tail["Is Laundering"].sum(),
      f"({tail['Is Laundering'].mean():.1%})")

# 2. Campaigns affected by the cutoff and the size filter
camp["crosses_cutoff"] = camp["end"] >= cut
camp["eval_ok"] = (camp["n_txn"] >= 3) & (camp["n_accounts"] >= 3)
print("Campaigns ending after cutoff:", camp["crosses_cutoff"].sum())
print("Campaigns passing size filter:", camp["eval_ok"].sum())

camp["split"] = np.select(
    [camp["start"] < "2022-09-06", camp["start"] < "2022-09-08"],
    ["train", "val"], default="test")
print(pd.crosstab(camp["split"], camp["eval_ok"]))

Tail rows: 1108 | laundering: 655 (59.1%)
Campaigns ending after cutoff: 109
Campaigns passing size filter: 258
eval_ok  False  True 
split                
test        37     76
train       53    126
val         22     56


In [7]:
print(pd.crosstab([camp["split"], camp["eval_ok"]], camp["crosses_cutoff"]))

crosses_cutoff  False  True 
split eval_ok               
test  False        27     10
      True          2     74
train False        53      0
      True        120      6
val   False        21      1
      True         38     18


In [8]:
camp["start_day"] = camp["start"].dt.date
print(pd.crosstab(camp.loc[camp.eval_ok, "start_day"], camp.loc[camp.eval_ok, "crosses_cutoff"],
                  margins=True))


crosses_cutoff  False  True  All
start_day                       
2022-09-01         31     0   31
2022-09-02         24     0   24
2022-09-03         20     0   20
2022-09-04         23     2   25
2022-09-05         22     4   26
2022-09-06         27     3   30
2022-09-07         11    15   26
2022-09-08          2    18   20
2022-09-09          0    28   28
2022-09-10          0    28   28
All               160    98  258


In [9]:
# 1. Group banks into 3 institutions, balanced by transaction volume
vol = pd.concat([df["From Bank"], df["To Bank"]]).value_counts()
K = 3
load = [0] * K
bank2inst = {}
for bank, n in vol.items():          # largest banks first
    k = load.index(min(load))        # give it to the lightest institution
    bank2inst[bank] = k
    load[k] += n
print("Institution loads:", load)

# 2. Natural visibility per campaign.
# An institution sees a transaction if the sender's or the receiver's bank belongs to it.
pat["inst_from"] = pat["From Bank"].map(bank2inst)
pat["inst_to"] = pat["To Bank"].map(bank2inst)

def max_visibility(g):
    return max(((g.inst_from == k) | (g.inst_to == k)).mean() for k in range(K))

camp["natural_vis"] = pat.groupby("campaign_id").apply(max_visibility, include_groups=False)
print(camp.loc[camp.eval_ok, "natural_vis"].describe().round(2))
print(pd.cut(camp.loc[camp.eval_ok, "natural_vis"], [0, .25, .5, .75, .99, 1]).value_counts().sort_index())

Institution loads: [3385564, 3385563, 3385563]
count    258.00
mean       0.88
std        0.14
min        0.50
25%        0.75
50%        1.00
75%        1.00
max        1.00
Name: natural_vis, dtype: float64
natural_vis
(0.0, 0.25]       0
(0.25, 0.5]       1
(0.5, 0.75]      73
(0.75, 0.99]     43
(0.99, 1.0]     141
Name: count, dtype: int64


In [10]:
hub = ["FAN-IN", "FAN-OUT", "GATHER-SCATTER"]
camp["group"] = np.where(camp["base_type"].isin(hub), "hub", "fragmentable")
camp["split2"] = pd.cut(camp["start"].dt.day, [0, 4, 5, 7, 10],
                        labels=["train", "val", "test", "stress"])

e = camp[camp.eval_ok]
print(pd.crosstab(e["split2"], e["group"], margins=True))
print(e.groupby("base_type")["natural_vis"].mean().round(2))

group   fragmentable  hub  All
split2                        
train             55   45  100
val               19    7   26
test              37   19   56
stress            41   35   76
All              152  106  258
base_type
BIPARTITE         0.76
CYCLE             0.80
FAN-IN            1.00
FAN-OUT           1.00
GATHER-SCATTER    1.00
RANDOM            0.83
SCATTER-GATHER    0.84
STACK             0.77
Name: natural_vis, dtype: float64


In [11]:
import random

K = 8
load = [0] * K
bank2inst = {}
for bank, n in vol.items():
    k = load.index(min(load)); bank2inst[bank] = k; load[k] += n

def vis_of(edges, assign):
    return max(np.mean([(assign[s] == k) or (assign[d] == k) for s, d in edges]) for k in range(K))

def reassign(edges, accounts, target, tries=300, rng=None):
    best, best_gap = None, 9
    for _ in range(tries):
        if target >= 0.999:
            k = rng.randrange(K); a = {x: k for x in accounts}
        else:
            m = rng.randint(2, K)                  # spread over m institutions
            insts = rng.sample(range(K), m)
            a = {x: rng.choice(insts) for x in accounts}
        v = vis_of(edges, a)
        if abs(v - target) < best_gap:
            best, best_gap = (a, v), abs(v - target)
        if best_gap < 0.03:
            break
    return best

frag = camp[(camp.eval_ok) & (camp.group == "fragmentable")].index
rng = random.Random(42)
rows = []
for target in [1.0, 0.75, 0.5, 0.25]:
    for cid in frag:
        g = pat[pat.campaign_id == cid]
        edges = list(zip(g.src, g.dst))
        accounts = set(g.src) | set(g.dst)
        _, v = reassign(edges, accounts, target, rng=rng)
        rows.append((target, cid, v))

res = pd.DataFrame(rows, columns=["target", "campaign_id", "achieved"])
print(res.groupby("target")["achieved"].describe().round(2))

# Accounts appearing in more than one campaign (their assignments could conflict)
acc_camps = pd.concat([pat[["campaign_id", "src"]].rename(columns={"src": "acc"}),
                       pat[["campaign_id", "dst"]].rename(columns={"dst": "acc"})]).drop_duplicates()
print("Accounts in >1 campaign:", (acc_camps.groupby("acc").size() > 1).sum())

        count  mean   std   min   25%   50%   75%  max
target                                                
0.25    152.0  0.42  0.13  0.25  0.32  0.38  0.50  1.0
0.50    152.0  0.53  0.08  0.40  0.50  0.50  0.55  1.0
0.75    152.0  0.74  0.05  0.50  0.71  0.75  0.77  1.0
1.00    152.0  1.00  0.00  1.00  1.00  1.00  1.00  1.0
Accounts in >1 campaign: 100


In [12]:
def local_search(edges, accounts, target, rng, iters=400):
    accounts = list(accounts)
    a = {x: rng.randrange(K) for x in accounts}
    v = vis_of(edges, a)
    for _ in range(iters):
        x = rng.choice(accounts); old = a[x]; a[x] = rng.randrange(K)
        nv = vis_of(edges, a)
        if abs(nv - target) <= abs(v - target): v = nv
        else: a[x] = old
    return a, v

rng = random.Random(42)
rows = []
for target in [1.0, 0.75, 0.5, 0.25]:
    for cid in frag:
        g = pat[pat.campaign_id == cid]
        edges = list(zip(g.src, g.dst)); accs = set(g.src) | set(g.dst)
        if target >= 0.999:
            v = 1.0
        else:
            _, v = local_search(edges, accs, target, rng)
        rows.append((target, cid, v))
res2 = pd.DataFrame(rows, columns=["target", "campaign_id", "achieved"])
print(res2.groupby("target")["achieved"].describe().round(2))

floor = res2[res2.target == 0.25].set_index("campaign_id")["achieved"]
print("Unfragmentable (floor >= 0.9):", (floor >= 0.9).sum())

# Shared accounts crossing splits
split_of = camp["split2"]
acc_split = acc_camps.assign(split=acc_camps.campaign_id.map(split_of))
cross = acc_split.groupby("acc")["split"].nunique()
print("Accounts shared across splits:", (cross > 1).sum())

        count  mean   std   min   25%   50%   75%  max
target                                                
0.25    152.0  0.38  0.15  0.20  0.27  0.33  0.50  1.0
0.50    152.0  0.53  0.08  0.40  0.50  0.50  0.55  1.0
0.75    152.0  0.74  0.05  0.67  0.71  0.74  0.75  1.0
1.00    152.0  1.00  0.00  1.00  1.00  1.00  1.00  1.0
Unfragmentable (floor >= 0.9): 2
Accounts shared across splits: 83


In [13]:
train_accs = set(acc_split.loc[acc_split.split == "train", "acc"])
test_camps = acc_split[acc_split.split == "test"]
tainted = test_camps[test_camps.acc.isin(train_accs)].campaign_id.unique()
ok_test = camp[(camp.split2 == "test") & camp.eval_ok & (camp.group == "fragmentable")].index
print("Fragmentable test campaigns sharing an account with train:",
      len(set(tainted) & set(ok_test)), "of", len(ok_test))

Fragmentable test campaigns sharing an account with train: 7 of 37
